# Cartoon plots of linear vs quadratic dynamics

In [ ]:
using Plots
using ColorSchemes
green_color = :green

In [ ]:
# Taken from analysis/tracks.jl
function onehit_cdf(t::Array, conc::Float64, 
        kappa::Float64, k2::Float64; t0::Float64=0.0)
    kc = kappa * conc
    tt0 = clamp.(t .- t0, 0.0, Inf)
    kdiff = (k2 - kc)
    fctpart = kc .* exp.(-k2 .* tt0) .- k2 .* exp.(-kc .* tt0)
    cdf = 1.0 .+ fctpart ./ kdiff
    return cdf
end

"""
CDF of the 2-hit model, coded in a way that prevents large
cancellation errors when tt0 is approximately 0. TODO
"""
function twohit_cdf_float(t::Float64, conc::Float64, 
        kappa::Float64, k2::Float64; t0::Float64=0.0)
    kc = kappa * conc
    kdiff = k2 - kc
    tt0 = clamp.(t .- t0, 0.0, Inf)
    if max(kc, k2) * tt0 < 1e-3
        # Taylor approximation to prevent large cancellation errors
        # Leading term is in t^3, so error is (1e-2)^4 < 1e-8, good
        cdf = k2 * kc * kc / 6.0 * tt0^3
    else
        prefact = k2 * kc / kdiff
        fact1 = (k2 - 2.0 * kc) / kdiff / kc
        fact2 = kc / kdiff / k2
        tt0ekct0 = tt0 * exp(-kc * tt0)
        term1 = fact1 * (1.0 - exp(-kc * tt0))
        term2 = fact2 * (1.0 - exp(-k2 * tt0))
        cdf = prefact * (term1 - tt0ekct0 + term2)
    end
    cdf = clamp(cdf, 1e-12, 1.0 - 1e-12)
    return cdf
end

function twohit_cdf(t::Array, conc::Float64, 
        kappa::Float64, k2::Float64; t0::Float64=0.0)
    return collect(map(
        x -> twohit_cdf_float(x, conc, kappa, k2; t0=t0), t))
end


""" CDF """
function threehit_cdf(t::Array, conc::Float64, 
        kappa::Float64, k2::Float64; t0::Float64=0.0)
    kc = kappa * conc
    tt0 = clamp.(t .- t0, 0.0, Inf)
    exptt0 = exp.(-kc .* tt0)
    kdif = k2 - kc
    cdf = -(k2 * (3.0*kc*kc - 3.0*kc*k2 + k2*k2) / kdif^3) .* exptt0
    cdf .+= ((kc * k2 * (2.0*kc - k2))/kdif^2) .* tt0 .* exptt0
    cdf .-= ((0.5 * kc*kc * k2) / kdif) .* (tt0 .* tt0) .* exptt0
    cdf .+= (kc / kdif)^3 .* exp.(-k2 .* tt0)
    cdf .+= 1.0
    cdf = clamp.(cdf, 1e-10, 1.0 - 1e-10)
    return cdf
end

In [ ]:
# Small 'cartoon' linear vs quadratic plots for publication panels
# Creates `panels/` and saves two PDF files (one per panel) and displays them inline
mkpath("panels")
# Time vector (coarse enough for a cartoon, fine enough for smooth lines)
t = collect(0:0.5:100)
# Plot full CDF in thin line, highlight early times with an inset
rate1 = 0.05
rate2 = rate1 * 2.0
rate3 = rate2 * 1.5
k2 = 2.0  # Fast cell-intrinsic activation step
cdf1 = onehit_cdf(t, 1.0, rate1, k2; t0=0.0)
cdf2 = twohit_cdf(t, 1.0, rate2, k2; t0=0.0)
cdf3 = threehit_cdf(t, 1.0, rate3, k2; t0=0.0)

fitfrac = 0.4
insetfrac = 0.05
tinset = collect(0:0.5*insetfrac:100*insetfrac)
oneinset = onehit_cdf(tinset, 1.0, rate1, k2; t0=0.0)
twoinset = twohit_cdf(tinset, 1.0, rate2, k2; t0=0.0)
threeinset = threehit_cdf(tinset, 1.0, rate3, k2; t0=0.0)

# Visual styling: small but legible fonts for labels/titles
label_font = font("Arial", 12)
title_font = font("Arial", 12)
ylbl = "Active cells"
lw = 2.0
plotsz = (210,160)
# Individual panels (small sizes, suitable for composite figures)
# Remove ticks/ticklabels and disable grid lines (xticks/yticks = false)
p1 = plot(t, cdf1,
          lw=lw, color=green_color,
          xlabel="Time", ylabel=ylbl,
          legend=false, #framestyle=:box, 
          xticks=false, yticks=false, xgrid=false, ygrid=false,
          guidefont=label_font, titlefont=title_font, size=plotsz)
plot!(p1,
    tinset, oneinset;
    inset = (1, bbox(0.05, 0.05, 0.4, 0.75, :bottom, :right)),
    ticks = nothing,
    subplot = 2,
    framestyle=:box,
    bg_inside = nothing, 
    label = nothing,
    lw=lw*2.0,
    color=green_color,
    aspect=:equal
)
# To mark where the inset was taken from, 
# define a function that returns a Plots.Shape
# https://discourse.julialang.org/t/how-to-draw-a-rectangular-region-with-plots-jl/1719/2
rectangle(w, h, x, y) = Shape(x .+ [0,w,w,0], y .+ [0,0,h,h])
rect1 = rectangle(t[end]*insetfrac*2, insetfrac*6,0,0)

plot!(p1, rect1, 
    opacity=1.0, lw=1.0, fillalpha=0.0, color=:transparent, linecolor=:black)

# Add small lines connecting the boxes
xx = maximum(rect1.x)
y1, y2 = extrema(rect1.y)
xinsbox = t[end] * (1.0 - 0.5)
yinsbox = 0.75
plot!(p1, [xx, xinsbox], [y1, 0.02], color=:black, lw=1.0, ls=:dash)
plot!(p1, [xx, xinsbox], [y2, yinsbox], color=:black, lw=1.0, ls=:dash)

# Annotate that the 1-hit model looks like 2-step because of cell activation step
annotate!(p1, xinsbox*1.3, yinsbox*0.8, (raw"$k_2$", :bottom, :center))
plot!(p1, [xinsbox*1.3, xinsbox*1.2], [yinsbox*0.6, 0.15],arrow=true,color=:black,linewidth=2)



# Two-hit model
p2 = plot(t, cdf2,
          lw=lw, color=green_color,
          xlabel="Time", ylabel=ylbl,
          legend=false, #framestyle=:box,
          xticks=false, yticks=false, xgrid=false, ygrid=false,
          guidefont=label_font, titlefont=title_font, size=plotsz)

plot!(p2,
    tinset, twoinset;
    inset = (1, bbox(0.05, 0.05, 0.4, 0.75, :bottom, :right)),
    ticks = nothing,
    subplot = 2,
    bg_inside = nothing, 
    label = nothing,
    framestyle=:box,
    lw=lw*2.0,
    color=green_color,
    aspect=:equal
)
rect2 = rectangle(t[end]*insetfrac*2, insetfrac*4,0,0)
plot!(p2, rect2, 
    opacity=1.0, lw=1.0, fillalpha=0.0, color=:transparent, linecolor=:black)
y1, y2 = extrema(rect2.y)
plot!(p2, [xx, xinsbox], [y1, 0.02], color=:black, lw=1.0, ls=:dash)
plot!(p2, [xx, xinsbox], [y2, yinsbox], color=:black, lw=1.0, ls=:dash)


# Also for a three-hit model
p3 = plot(t, cdf3,
          lw=lw, color=green_color,
          xlabel="Time", ylabel=ylbl,
          legend=false, #framestyle=:box,
          xticks=false, yticks=false, xgrid=false, ygrid=false,
          guidefont=label_font, titlefont=title_font, size=plotsz)

plot!(p3,
    tinset, threeinset;
    inset = (1, bbox(0.05, 0.05, 0.4, 0.75, :bottom, :right)),
    ticks = nothing,
    subplot = 2,
    bg_inside = nothing, 
    label = nothing,
    framestyle=:box,
    lw=lw*2.0,
    color=green_color,
    aspect=:equal
)
rect3 = rectangle(t[end]*insetfrac*2, insetfrac*4,0,0)
plot!(p3, rect3, 
    opacity=1.0, lw=1.0, fillalpha=0.0, color=:transparent, linecolor=:black)
y1, y2 = extrema(rect3.y)
plot!(p3, [xx, xinsbox], [y1, 0.02], color=:black, lw=1.0, ls=:dash)
plot!(p3, [xx, xinsbox], [y2, yinsbox], color=:black, lw=1.0, ls=:dash)


# Display each panel in the notebook
display(p1)
display(p2)
display(p3)

# Save each panel as a PDF (no combined plot)
savefig(p1, "figures/final_tracks_panels/one-hit_cartoon.pdf")
savefig(p2, "figures/final_tracks_panels/two-hit_cartoon.pdf")
savefig(p3, "figures/final_tracks_panels/three-hit_cartoon.pdf")

# Print saved file paths for quick confirmation
println("Saved: figures/final_tracks_panels/one-hit_cartoon.pdf, "
        *"figures/final_tracks_panels/two-hit_cartoon.pdf, "
        *"figures/final_tracks_panels/three-hit_cartoon.pdf")

# Comparison of full CDF curves adjusted to have the same $k_\mathrm{obs}$

The different models can capture the same overall $k_\mathrm{obs}$, but this entails different overall curve shapes; they are not distinguished solely at early times, but by all events. 

In [1]:
# TODO: decide if I want to do this or not, maybe not worth the time. 